In [1]:
%pip install pandas
%pip install pyarrow
%pip install statsmodels
%pip install scipy

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import os
import pyarrow
import numpy as np
from scipy.signal import lfilter
from statsmodels.tsa.stattools import adfuller
import threading

In [3]:
def get_files():
    files = os.listdir("./dollar_bars_parquet")
    files = ["./dollar_bars_parquet/" + file for file in files]
    return files

files = get_files()

# Goals

Our data at the moment abandoned our idea of time bars for Dollar Bars, solving the issue of non-constant information arrival. 

This gets us to a great baseline, but we need to solve another large issue with our baseline data.

As it stands right now, we hopefully have normally distributed (Gaussian) bars but they are non-stationary



In [ ]:
input_dir = "./dollar_bars_parquet"
output_dir = "./dollar_bars_fracdiff_parquet"
os.makedirs(output_dir, exist_ok=True)

def safe_read_parquet(path: str) -> pd.DataFrame:
    try:
        return pd.read_parquet(path, engine="pyarrow")
    except Exception:
        import pyarrow.parquet as pq
        table = pq.read_table(path, arrow_extensions_enabled=False)
        return table.to_pandas()

def get_weights_ffd(d: float, thresh: float = 1e-5) -> np.ndarray:
    w = [1.0]
    k = 1
    while True:
        wk = -w[-1] * (d - k + 1) / k
        if abs(wk) < thresh:
            break
        w.append(wk)
        k += 1
    return np.array(w, dtype=np.float64)

def frac_diff_ffd_fast(series: pd.Series, d: float, thresh: float = 1e-5) -> pd.Series:
    from scipy.signal import lfilter
    x = pd.to_numeric(series, errors="coerce").astype(float).ffill()
    w = get_weights_ffd(d, thresh=thresh)
    out_vals = lfilter(w, 1.0, x.values)
    out = pd.Series(out_vals, index=x.index, dtype=np.float64)

    width = len(w) - 1
    out.iloc[:width] = np.nan
    return out

def find_min_d_adf(
    series: pd.Series,
    d_start: float = 0.10,
    d_end: float = 1.00,
    d_step: float = 0.05,       # ← was 0.01, use coarser step first
    p_thresh: float = 0.01,
    ffd_thresh: float = 1e-5,
    max_rows: int = 20_000,     # ← NEW: cap ADF input size
):
    s = pd.to_numeric(series, errors="coerce").astype(float).ffill().dropna()
    best = None

    max_w = get_weights_ffd(d_start, thresh=ffd_thresh)
    max_window = len(max_w) - 1

    if len(s) <= max_window + 30:
        return None 

    for d in np.round(np.arange(d_start, d_end + d_step, d_step), 2):
        fd = frac_diff_ffd_fast(s, d=float(d), thresh=ffd_thresh)
        fd_test = fd.iloc[max_window:].dropna()

        if len(fd_test) < 30:
            continue

        # ── cap rows so adfuller doesn't take forever ────────────────
        if len(fd_test) > max_rows:
            fd_test = fd_test.iloc[-max_rows:]

        try:
            # maxlag cap prevents AIC from searching too many lags
            max_lag = min(30, len(fd_test) // 10)
            pval = adfuller(fd_test.values, maxlag=max_lag, autolag="AIC")[1]
        except Exception:
            continue

        print(f"  d={d:.2f}  ADF p={pval:.4e}")   # progress feedback

        if pval < p_thresh:
            best = {"d": float(d), "pvalue": float(pval)}
            break

    return best

def process_single_file(file_name: str, input_dir: str, output_dir: str):
    try:
        df = safe_read_parquet(file_name)
        if "start_ts" in df.columns:
            df["start_ts"] = pd.to_datetime(df["start_ts"], errors="coerce")
        if "end_ts" in df.columns:
            df["end_ts"] = pd.to_datetime(df["end_ts"], errors="coerce")

        df["_row_id"] = np.arange(len(df))
        sort_cols = [c for c in ["start_ts", "end_ts", "_row_id"] if c in df.columns]
        if sort_cols:
            df = df.sort_values(sort_cols).reset_index(drop=True)

        if "close" not in df.columns:
            print(f"Skipping {file_name}: missing close column")
            return

        result = find_min_d_adf(df["close"])
        if result is not None:
            best_d = result["d"]
            best_p = result["pvalue"]
            col = f"fd_close_d{str(best_d).replace('.', '_')}"
            df[col] = frac_diff_ffd_fast(df["close"], d=best_d)
            df["best_d"] = best_d
            df["best_pvalue"] = best_p
        else:
            df["fd_close_d_none"] = np.nan
            df["best_d"] = np.nan
            df["best_pvalue"] = np.nan

        df = df.drop(columns=["_row_id"], errors="ignore")
        out_path = os.path.join(output_dir, os.path.basename(file_name))
        df.to_parquet(out_path, index=False)
        print(f"Saved {out_path}")
    except Exception as e:
        print(f"Error processing {file_name}: {e}")

def frac_dif_adder_threaded(num_threads: int = 4):
    import threading
    from queue import Queue

    # Create a queue of files
    file_queue = Queue()
    for file_name in files:
        file_queue.put(file_name)

    def worker():
        while True:
            file_name = file_queue.get()
            if file_name is None:
                break
            process_single_file(file_name, input_dir, output_dir)
            file_queue.task_done()

    # Create and start worker threads
    threads = []
    for _ in range(num_threads):
        t = threading.Thread(target=worker, daemon=False)
        t.start()
        threads.append(t)

    # Wait for all files to be processed
    file_queue.join()

    # Send sentinel values to stop workers
    for _ in range(num_threads):
        file_queue.put(None)

    # Wait for all threads to finish
    for t in threads:
        t.join()

    print(f"All {len(files)} files processed.")

frac_dif_adder_threaded(num_threads=4)

  d=0.10  ADF p=1.3813e-01
  d=0.10  ADF p=6.4833e-01
  d=0.10  ADF p=2.2787e-01
  d=0.10  ADF p=4.3178e-03
Saved ./dollar_bars_fracdiff_parquet\AAOI.parquet
  d=0.15  ADF p=4.1443e-02
  d=0.15  ADF p=7.4690e-02
  d=0.15  ADF p=3.9408e-01
  d=0.10  ADF p=2.4444e-01
  d=0.20  ADF p=5.2101e-03
  d=0.20  ADF p=1.6839e-01
  d=0.20  ADF p=2.3569e-02
Saved ./dollar_bars_fracdiff_parquet\AAL.parquet
  d=0.15  ADF p=1.2360e-01
  d=0.25  ADF p=3.6389e-02
  d=0.25  ADF p=2.7480e-03
Saved ./dollar_bars_fracdiff_parquet\A.parquet
  d=0.10  ADF p=9.7192e-02
  d=0.20  ADF p=2.4222e-02
  d=0.30  ADF p=3.1279e-03
Saved ./dollar_bars_fracdiff_parquet\AA.parquet
  d=0.15  ADF p=2.4249e-02
  d=0.10  ADF p=3.3519e-03
  d=0.25  ADF p=1.8686e-03
Saved ./dollar_bars_fracdiff_parquet\ABNB.parquet
Saved ./dollar_bars_fracdiff_parquet\AAPL.parquet
  d=0.20  ADF p=2.6401e-03
  d=0.10  ADF p=3.1637e-02
Saved ./dollar_bars_fracdiff_parquet\ABBV.parquet
  d=0.10  ADF p=1.6084e-01
  d=0.10  ADF p=4.4360e-06
Saved ./

: 